# Tahap 4: Konfirmasi Bahan Pelengkap + Jawaban Natural via Llama3 (Ollama)

Lanjutan dari `03_reranking.ipynb`.

Alur di notebook ini:
1. Setelah user input bahan awal & dapat hasil rerank awal (dari notebook 03), sistem cek: dari daftar **bahan dasar dapur umum** (garam, minyak goreng, bawang, dll), mana saja yang BELUM disebut user tapi relevan/sering dipakai.
2. Llama3 (via Ollama) menyusun **pertanyaan konfirmasi yang natural & tidak template** berdasarkan daftar itu -- bukan nge-print daftar mentah, tapi ngobrol menyesuaikan bahan yang sudah diketik user.
3. User menjawab bebas (mis. "aku ada bawang putih sama kecap manis doang", atau "cuma itu doang").
4. Llama3 mengekstrak jawaban user itu jadi daftar bahan terstruktur (JSON) -- termasuk bisa mengenali kalau user bilang tidak ada tambahan.
5. Bahan tambahan digabung ke `user_ingredients`, lalu **rerank ulang total** (retrieve SBERT + hitung ulang efficiency score) pakai `rerank()` dari notebook 03.
6. Llama3 menyusun jawaban akhir yang natural, dari hasil rerank final -- tapi grounded (tidak boleh mengarang bahan/skor yang tidak ada di data).

In [1]:
# !pip install ollama -q

In [2]:
# !pip install nbformat

In [3]:
# !pip install ollama -q
# Pastikan Ollama sudah jalan lokal: `ollama serve`, dan modelnya sudah ditarik: `ollama pull llama3`

import json
import ollama

LLM_MODEL = "llama3"

## 1. Import Fungsi dari Notebook Sebelumnya

Supaya tidak duplikat kode, idealnya `parse_user_ingredients`, `retrieve_candidates`, `compute_efficiency_score`, dan `rerank` dipindah ke modul `.py` bersama (mis. `recipe_engine.py`) lalu di-import di sini maupun di notebook 03. Untuk sementara notebook ini asumsikan kamu sudah punya fungsi-fungsi itu tersedia (copy dari `03_reranking.ipynb`, atau `%run 03_reranking.ipynb` kalau mau langsung eksekusi semua sel di atasnya).

In [4]:
# %run 03_reranking.ipynb
# (opsional, kalau mau langsung load semua fungsi + koneksi ChromaDB dari notebook 03)

In [5]:
import re
import html
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "resep_masakan"

model = SentenceTransformer(MODEL_NAME)
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)

print("Jumlah dokumen di collection:", collection.count())


def add_query_prefix(text: str) -> str:
    return f"query: {text}"


def add_passage_prefix(text: str) -> str:
    return f"passage: {text}"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5051.36it/s]


Jumlah dokumen di collection: 5000


In [7]:
UNITS = [
    "sendok makan",
    "sendok teh",
    "sdm",
    "sdt",
    "gram",
    "gr",
    "kg",
    "ons",
    "ml",
    "mili",
    "liter",
    "ltr",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ikat",
    "ruas",
    "cm",
    "potong",
    "ptg",
    "biji",
    "bj",
    "ekor",
    "ekr",
    "genggam",
    "bungkus",
    "bks",
    "sachet",
    "sct",
    "papan",
    "buku",
    "helai",
    "hlai",
    "gelas",
    "gls",
    "cup",
    "mangkuk",
    "piring",
    "sisir",
    "kotak",
    "ktk",
    "kaleng",
    "klg",
    "botol",
    "btl",
]
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)
QTY_PATTERN = (
    r"\d+\s*-\s*\d+"
    r"|\d+\s+\d+\s*/\s*\d+"
    r"|\d+\s*/\s*\d+"
    r"|\d+[.,]\d+"
    r"|\d+"
    r"|sesuai selera"
    r"|secukupnya"
    r"|sejumput"
    r"|beberapa"
)
# FIX: \b sebelum unit dihapus -> '100gr' (tanpa spasi) sekarang terdeteksi unitnya dgn benar
# (\d dan huruf sama2 word-char di regex, jadi '100'|'gr' tidak punya boundary di antaranya).
INGREDIENT_REGEX = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?P<unit>{UNIT_PATTERN})?\b\.?\s*(?P<name>.+?)\s*$",
    re.IGNORECASE,
)


def clean_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"\(.*?\)", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def parse_ingredient(raw: str) -> dict:
    raw = html.unescape(raw).strip()
    raw = re.sub(r"^[\-\*•]+\s*", "", raw)
    # FIX: split hanya di koma yang TIDAK diapit angka (supaya '1,5 sdm ...' -> desimal, bukan terpotong)
    split_match = re.search(r"(?<!\d),(?!\d)", raw)
    if split_match:
        main, note = raw[: split_match.start()], raw[split_match.start() + 1 :]
    else:
        main, note = raw, ""
    main = main.strip()
    m = INGREDIENT_REGEX.match(main)
    if m and m.group("name"):
        name = clean_name(m.group("name"))
    else:
        name = clean_name(main)
    return {"raw": raw, "name": name}


def parse_user_ingredients(text: str) -> list:
    """Input: '2 butir telur, 100gr dada ayam, 2 siung bawang putih'
    Note: user memisahkan tiap bahan dgn koma, TAPI parse_ingredient juga memakai
    koma untuk memisahkan catatan (mis. 'dikocok lepas'). Untuk input user yang
    berupa daftar bahan bersih (tanpa catatan cara masak), kita split dulu per
    kemungkinan bahan pakai heuristik: pisah di koma yang diikuti oleh angka/qty
    word atau di awal token baru. Untuk kasus sederhana (tanpa catatan), split
    koma biasa sudah cukup."""
    items = [t.strip() for t in text.split(",") if t.strip()]
    parsed = [parse_ingredient(item) for item in items]
    return [p["name"] for p in parsed if p["name"]]


# test (termasuk kasus '100gr' tanpa spasi yang sebelumnya bug)
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
user_ingredients = parse_user_ingredients(user_input)
print(user_ingredients)  # harus: ['telur', 'dada ayam', 'bawang putih']

['telur', 'dada ayam', 'bawang putih']


In [8]:
def ingredient_matches(user_name: str, recipe_name: str) -> bool:
    u = user_name.lower().strip()
    r = recipe_name.lower().strip()
    if not u or not r:
        return False
    return u in r or r in u


def compute_efficiency_score(user_ingredients: list, recipe_ingredients: list) -> dict:
    if not recipe_ingredients:
        return {
            "matched": 0,
            "coverage_recipe": 0.0,
            "coverage_user": 0.0,
            "efficiency_score": 0.0,
            "matched_items": [],
            "missing_items": [],
        }

    matched_recipe_items = []
    matched_user_items = set()
    for r_name in recipe_ingredients:
        hit = False
        for u_name in user_ingredients:
            if ingredient_matches(u_name, r_name):
                hit = True
                matched_user_items.add(u_name)
        if hit:
            matched_recipe_items.append(r_name)

    matched = len(matched_recipe_items)
    missing_items = [r for r in recipe_ingredients if r not in matched_recipe_items]

    coverage_recipe = matched / len(recipe_ingredients)
    coverage_user = (
        len(matched_user_items) / len(user_ingredients) if user_ingredients else 0.0
    )

    if coverage_recipe + coverage_user == 0:
        efficiency_score = 0.0
    else:
        efficiency_score = (2 * coverage_recipe * coverage_user) / (
            coverage_recipe + coverage_user
        )

    return {
        "matched": matched,
        "coverage_recipe": coverage_recipe,
        "coverage_user": coverage_user,
        "efficiency_score": efficiency_score,
        "matched_items": matched_recipe_items,
        "missing_items": missing_items,
    }


# test cepat
recipe_ex = ["telur ayam", "dada ayam fillet", "bawang putih", "kecap manis", "garam"]
print(compute_efficiency_score(user_ingredients, recipe_ex))

{'matched': 3, 'coverage_recipe': 0.6, 'coverage_user': 1.0, 'efficiency_score': 0.7499999999999999, 'matched_items': ['telur ayam', 'dada ayam fillet', 'bawang putih'], 'missing_items': ['kecap manis', 'garam']}


In [9]:
def retrieve_candidates(user_text: str, top_k: int = 30) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    candidates = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        candidates.append(
            {
                "id": results["ids"][0][i],
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "steps_text": meta["steps_text"],
                "sbert_score": 1
                - results["distances"][0][i],  # cosine distance -> similarity
            }
        )
    return candidates


def rerank(
    user_text: str,
    top_k_candidates: int = 30,
    top_n_final: int = 5,
    w_sbert: float = 0.5,
    w_efficiency: float = 0.5,
) -> list:
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates(user_text, top_k=top_k_candidates)

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        eff = compute_efficiency_score(user_ingredients, c["ingredient_names"])
        c.update(eff)
        c["sbert_score_norm"] = float(s_norm)
        c["final_score"] = (
            w_sbert * float(s_norm) + w_efficiency * eff["efficiency_score"]
        )

    candidates_sorted = sorted(candidates, key=lambda c: c["final_score"], reverse=True)
    return candidates_sorted[:top_n_final], user_ingredients

## 2. Daftar Bahan Dasar Dapur Umum

Ini bahan-bahan yang **sering dipakai tapi jarang diketik user** karena dianggap "pasti ada". Kita cek mana dari daftar ini yang belum disebut user, lalu itu jadi bahan kandidat untuk ditanyakan (bukan diambil dari `missing_items` resep, sesuai keputusan kamu).

In [10]:
PANTRY_STAPLES = [
    "garam",
    "gula",
    "gula pasir",
    "merica",
    "merica bubuk",
    "lada",
    "minyak goreng",
    "kecap manis",
    "kecap asin",
    "saus tiram",
    "bawang merah",
    "bawang putih",
    "daun bawang",
    "cabai",
    "cabe rawit",
    "penyedap rasa",
    "kaldu bubuk",
    "air",
    "tepung terigu",
    "tepung maizena",
    "daun salam",
    "daun jeruk",
    "jahe",
    "kunyit",
    "lengkuas",
    "ketumbar",
]


def ingredient_matches(user_name: str, ref_name: str) -> bool:
    u, r = user_name.lower().strip(), ref_name.lower().strip()
    return bool(u) and bool(r) and (u in r or r in u)


def get_unmentioned_staples(user_ingredients: list) -> list:
    """Bahan dasar dapur yang belum disinggung user sama sekali."""
    unmentioned = []
    for staple in PANTRY_STAPLES:
        if not any(ingredient_matches(u, staple) for u in user_ingredients):
            unmentioned.append(staple)
    return unmentioned


# test
contoh_user = ["telur", "dada ayam", "bawang putih"]
print(get_unmentioned_staples(contoh_user))

['garam', 'gula', 'gula pasir', 'merica', 'merica bubuk', 'lada', 'minyak goreng', 'kecap manis', 'kecap asin', 'saus tiram', 'bawang merah', 'daun bawang', 'cabai', 'cabe rawit', 'penyedap rasa', 'kaldu bubuk', 'air', 'tepung terigu', 'tepung maizena', 'daun salam', 'daun jeruk', 'jahe', 'kunyit', 'lengkuas', 'ketumbar']


## 3. Pertanyaan Konfirmasi yang Natural (bukan template)

Daripada nge-print `"Apakah kamu juga punya: garam, gula, minyak goreng, ...?"` mentah-mentah tiap kali (kelihatan robotic & template), kita minta Llama3 yang menyusun kalimatnya -- dengan `temperature` agak tinggi supaya frasanya bervariasi tiap dipanggil, dan tetap nyambung ke bahan yang sudah diketik user.

In [11]:
def generate_clarifying_question(
    user_text: str, user_ingredients: list, unmentioned_staples: list
) -> str:
    """Minta Llama3 membuat 1 pertanyaan konfirmasi yang natural dalam Bahasa Indonesia,
    tentang apakah user juga punya bahan-bahan dapur umum di `unmentioned_staples`.
    Tidak template -- LLM bebas milih gaya/urutan/kalimat, TIDAK boleh menambah bahan
    di luar daftar yang diberikan."""

    if not unmentioned_staples:
        return None  # semua bahan dasar udah disebut user, gak perlu nanya lagi

    system_prompt = (
        "Kamu adalah asisten resep masakan yang ramah dan ngobrol santai dalam Bahasa Indonesia. "
        "Tugasmu: tanyakan ke user apakah mereka juga punya sebagian dari daftar bahan dapur umum "
        "berikut, supaya rekomendasi resep bisa lebih pas. JANGAN menyebutkan semua bahan dalam daftar "
        "sekaligus seperti daftar belanja -- pilih beberapa yang paling masuk akal/relevan dengan bahan "
        "yang sudah disebut user, dan tanyakan dengan kalimat yang mengalir, bukan format list. "
        "Jangan pernah menyebut bahan di luar daftar yang diberikan. Balas HANYA dengan pertanyaannya "
        "saja (1-2 kalimat), tanpa basa-basi pembuka seperti 'Tentu,' atau 'Baik,'."
    )
    user_prompt = (
        f"Bahan yang sudah disebut user: {', '.join(user_ingredients)}.\n"
        f'Input asli user: "{user_text}"\n'
        f"Daftar bahan dapur umum yang belum disebut: {', '.join(unmentioned_staples)}."
    )

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0.9},
    )
    return response["message"]["content"].strip()


# test
user_text = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
user_ingredients = ["telur", "dada ayam", "bawang putih"]
unmentioned = get_unmentioned_staples(user_ingredients)

question = generate_clarifying_question(user_text, user_ingredients, unmentioned)
print(question)

Kamu punya bumbu-bumbuan lainnya di dapur, kayak apa ya? Misalnya, kamu juga punya garam dan merica, atau mungkin minyak goreng untuk menggoreng?


## 4. Ekstrak Jawaban User Jadi Bahan Terstruktur

User bisa jawab macam-macam: `"ada garam sama minyak goreng doang"`, `"lengkap semua kayaknya"`, `"nggak ada tambahan"`, dll. Kita minta Llama3 mengekstrak itu jadi JSON, **dibatasi hanya ke bahan yang memang ada di `unmentioned_staples`** (supaya tidak halusinasi nambahin bahan yang tidak pernah disebut user).

In [12]:
def parse_user_confirmation(user_reply: str, unmentioned_staples: list) -> list:
    """Ekstrak bahan yang dikonfirmasi user DARI daftar unmentioned_staples saja.
    Return: list nama bahan (subset dari unmentioned_staples)."""

    system_prompt = (
        "Kamu mengekstrak informasi dari jawaban user tentang bahan masakan yang mereka punya. "
        "Kamu HANYA boleh memilih dari daftar kandidat yang diberikan -- jangan menambah bahan lain, "
        "jangan menerka. Kalau user bilang tidak ada tambahan / cuma itu saja / dsb, kembalikan list kosong. "
        'Balas HANYA dengan JSON valid, format persis: {"confirmed_ingredients": ["...", ...]}. '
        "Tanpa penjelasan atau teks lain di luar JSON."
    )
    user_prompt = (
        f"Daftar kandidat bahan: {json.dumps(unmentioned_staples, ensure_ascii=False)}\n"
        f'Jawaban user: "{user_reply}"'
    )

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        format="json",  # paksa Ollama keluarin JSON valid (didukung Llama3 di Ollama)
        options={"temperature": 0.2},  # rendah -- ini tugas ekstraksi, bukan kreatif
    )

    try:
        parsed = json.loads(response["message"]["content"])
        confirmed = parsed.get("confirmed_ingredients", [])
    except (json.JSONDecodeError, AttributeError):
        confirmed = []

    # safety net: filter ulang, pastikan hasil LLM benar2 subset dari kandidat (anti-halusinasi)
    confirmed = [
        c
        for c in confirmed
        if any(ingredient_matches(c, cand) for cand in unmentioned_staples)
    ]
    return confirmed


# test
user_reply = "aku ada garam sama minyak goreng doang kayaknya"
confirmed = parse_user_confirmation(user_reply, unmentioned)
print(confirmed)

['garam', 'minyak goreng']


## 5. Gabung Bahan & Rerank Ulang Total

Bahan yang dikonfirmasi digabung ke `user_ingredients`, lalu panggil ulang `rerank()` dari notebook 03 -- retrieve SBERT ulang + hitung ulang efficiency score dari awal (bukan cuma menyaring top-5 lama), sesuai keputusanmu.

In [13]:
def rerank_with_confirmation(
    user_text: str,
    user_ingredients: list,
    confirmed_extra: list,
    top_k_candidates: int = 30,
    top_n_final: int = 5,
):
    """Gabungkan bahan awal + bahan tambahan yang dikonfirmasi, lalu rerank ulang total.
    Catatan: rerank() dari notebook 03 mem-parsing ulang user_text mentah -- karena bahan
    tambahan tidak ada di user_text asli, kita panggil versi internalnya langsung supaya
    tidak perlu 'mengarang' ulang kalimat user."""

    final_user_ingredients = list(
        dict.fromkeys(user_ingredients + confirmed_extra)
    )  # gabung, buang duplikat

    candidates = retrieve_candidates(user_text, top_k=top_k_candidates)

    import numpy as np

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        eff = compute_efficiency_score(final_user_ingredients, c["ingredient_names"])
        c.update(eff)
        c["sbert_score_norm"] = float(s_norm)
        c["final_score"] = 0.5 * float(s_norm) + 0.5 * eff["efficiency_score"]

    candidates_sorted = sorted(candidates, key=lambda c: c["final_score"], reverse=True)
    return candidates_sorted[:top_n_final], final_user_ingredients


# test
final_results, final_user_ingredients = rerank_with_confirmation(
    user_text, user_ingredients, confirmed
)
print("Bahan final:", final_user_ingredients)
for c in final_results:
    print(c["title"], round(c["final_score"], 3))

Bahan final: ['telur', 'dada ayam', 'bawang putih', 'garam', 'minyak goreng']
Telur Dadar Daun Bawang 0.7
Telur Dadar 0.644
Telur Dadar Tempe 0.584
Telur Dadar Tempe 0.545
Telur Dadar 0.538


## 6. Jawaban Akhir yang Natural (Grounded, Tidak Halusinasi)

Llama3 menulis rekomendasi dengan gaya ngobrol, TAPI semua fakta (judul, bahan yang kurang, dst) dikirim eksplisit di prompt -- jadi LLM tinggal merangkai kalimat, bukan mengarang data resep.

In [14]:
def generate_final_response(final_results: list, final_user_ingredients: list) -> str:
    facts = [
        {
            "judul": c["title"],
            "bahan_sudah_ada": c["matched_items"],
            "bahan_perlu_ditambah": c["missing_items"],
        }
        for c in final_results
    ]

    system_prompt = (
        "Kamu asisten resep masakan yang ramah, ngobrol natural dalam Bahasa Indonesia -- bukan robot "
        "yang nge-print list template. Susun rekomendasi dari data resep yang diberikan (JSON). "
        "ATURAN PENTING: jangan mengarang bahan, judul, atau langkah masak yang tidak ada di data. "
        "Sebutkan tiap resep, bahan yang perlu ditambah user, dan boleh kasih 1 kalimat alasan kenapa "
        "itu cocok (misal karena bahan pelengkapnya cuma sedikit). Jangan gunakan format list bernomor "
        "yang kaku -- rangkai jadi paragraf ngobrol."
    )
    user_prompt = (
        f"Bahan yang dipunya user: {', '.join(final_user_ingredients)}.\n"
        f"Data resep hasil sistem (JSON, urutan = urutan rekomendasi):\n{json.dumps(facts, ensure_ascii=False, indent=2)}"
    )

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0.8},
    )
    return response["message"]["content"].strip()


final_message = generate_final_response(final_results, final_user_ingredients)
print(final_message)

Kabar baik! Kita punya beberapa resep telur dadar yang sesuai dengan bahan-bahan yang sudah ada di rumah. Berikut rekomendasi dari data resep:

Kita dapat membuat "Telur Dadar Daun Bawang" dan hanya perlu menambahkan daun bawang merah, cabai, serta merica bubuk. Cepat dan mudah, kan? Selain itu, kita juga dapat mencoba "Telur Dadar" yang menggunakan bahan-bahan seperti bawang merah, daun bawang, dan minyak goreng. Keduanya sesuai dengan bahan telur, dada ayam, bawang putih, garam, dan minyak goreng yang sudah kita punya.

Saya tidak merekomendasikan "Telur Dadar Tempe" karena memerlukan tempe, daun jeruk, serta beberapa bumbu lainnya yang belum ada di rumah. Sama halnya dengan "Telur Dadar" yang terakhir, memerlukan kornet sapi, keju cheddar, dan lada bubuk yang tidak tersedia.

Jadi, pilihan-pilihan kita adalah "Telur Dadar Daun Bawang" atau "Telur Dadar". Keduanya mudah dibuat dan menggunakan bahan-bahan yang sudah kita punya.


## 7. Rangkai Jadi Satu Alur Chatbot

Simulasi alur end-to-end: input awal -> retrieval+rerank awal (dari 03) -> tanya bahan pelengkap dari `PANTRY_STAPLES` -> user jawab -> rerank ulang total -> jawaban akhir natural.

In [15]:
def chatbot_flow(user_text: str, user_reply_to_confirmation: str):
    # 1) parsing bahan awal (dari notebook 03)
    user_ingredients = parse_user_ingredients(user_text)

    # 2) cek bahan dasar dapur yang belum disebut
    unmentioned = get_unmentioned_staples(user_ingredients)

    # 3) LLM susun pertanyaan konfirmasi (kalau ada yang perlu ditanyakan)
    question = generate_clarifying_question(user_text, user_ingredients, unmentioned)
    if question:
        print("BOT:", question)
        print("USER:", user_reply_to_confirmation)

    # 4) ekstrak jawaban user
    confirmed_extra = (
        parse_user_confirmation(user_reply_to_confirmation, unmentioned)
        if question
        else []
    )

    # 5) rerank ulang total
    final_results, final_user_ingredients = rerank_with_confirmation(
        user_text, user_ingredients, confirmed_extra
    )

    # 6) jawaban akhir natural
    final_message = generate_final_response(final_results, final_user_ingredients)
    print("\nBOT:", final_message)
    return final_results, final_user_ingredients, final_message


# contoh pemakaian
chatbot_flow(
    user_text="2 butir telur, 100gr dada ayam, 2 siung bawang putih",
    user_reply_to_confirmation="ada garam sama minyak goreng doang kayaknya, sisanya nggak ada",
)

BOT: Kamu punya garam? Dan apakah kamu juga punya minyak goreng?
USER: ada garam sama minyak goreng doang kayaknya, sisanya nggak ada

BOT: Kebetulan user punya bahan-bahan yang cukup lengkap! Kita akan berdiskusi tentang beberapa resep telur dadar yang cocok dengan apa yang kamu miliki.

Pertama, kalian pasti pernah mendengar tentang "Telur Dadar Daun Bawang". Hmm, sayangnya, kalian tidak punya daun bawang merah dan cabai, tapi garam sudah ada! Jadi, kita hanya butuh beberapa bahan pelengkap untuk menyelesaikan resep ini. Saya merekomendasikan kalian mencoba tambahkan sedikit daun bawang merah dan cabai untuk memberi sentuhan rasa segar pada telur dadar.

Kedua, "Telur Dadar" adalah resep klasik yang pasti cocok dengan apa yang kamu punya. Telur, bawang putih, garam, dan minyak goreng sudah ada di kabinetmu! Kita hanya butuh sedikit bawang merah dan daun bawang untuk menyelesaikan resep ini.

Ketiga, "Telur Dadar Tempe" adalah variasi yang menarik. Sayangnya, kita tidak punya tempe, t

([{'id': 'resep_2582',
   'title': 'Telur Dadar Daun Bawang',
   'ingredient_names': ['telur ayam',
    'daun bawang merah',
    'cabai',
    'garam',
    'merica bubuk'],
   'steps_text': '1. Pisahkan Bawang merah dengan daunnya. Iris tipis bawang merah. Iris anjang daun bawang. Rajang cabe. 2. Kocok telur bersama garam, merica, kemudian masukkan daun bawang, bawang merah dna cabai. 3. Panaskan minyak di wajan, tuang adonan telur. Masak dengan api sedang hingga kedua sisi matang.',
   'sbert_score': 0.7709106206893921,
   'matched': 2,
   'coverage_recipe': 0.4,
   'coverage_user': 0.4,
   'efficiency_score': 0.4000000000000001,
   'matched_items': ['telur ayam', 'garam'],
   'missing_items': ['daun bawang merah', 'cabai', 'merica bubuk'],
   'sbert_score_norm': 0.9999999915642659,
   'final_score': 0.699999995782133},
  {'id': 'resep_4346',
   'title': 'Telur Dadar',
   'ingredient_names': ['telur',
    'bawang merah',
    'bawang putih',
    'daun bawang',
    'garam dan penyedap',


## Catatan & Yang Perlu Kamu Sesuaikan

- Notebook ini **belum bisa dijalankan langsung** di environment aku (tidak ada akses Ollama/model llama3 di sini) -- coba jalankan di environment kamu sendiri yang sudah ada Ollama + model llama3.
- `PANTRY_STAPLES` masih daftar manual (baseline) -- kalau mau lebih akurat, bisa dihitung dari data: cari bahan yang paling sering muncul di seluruh dataset (`ingredient_names`), itu kandidat "bahan dasar" yang lebih data-driven daripada nebak manual.
- Fungsi `retrieve_candidates` dan `compute_efficiency_score` dipakai dari notebook 03 -- sebaiknya sekarang benar-benar dipindah ke modul `.py` bersama (mis. `recipe_engine.py`), supaya notebook 03 dan 04 tidak duplikat kode lagi.
- `format="json"` di `ollama.chat` memaksa Llama3 keluar JSON valid -- ini fitur structured output Ollama, cukup andal tapi tetap ada filter subset-check di `parse_user_confirmation` sebagai pengaman anti-halusinasi.
- Kalau nanti mau dites akurasinya, ide sederhana: siapkan beberapa skenario input + jawaban konfirmasi manual, cek apakah `confirmed_extra` yang diekstrak sesuai ekspektasi, dan apakah `final_results` berubah urutannya secara masuk akal dibanding sebelum konfirmasi.